# JPM Regression RV Query-Driven Backtest

Systematic relative value strategy for USD SOFR butterflies using rolling OLS regression,
frozen out-of-sample residuals, directional stop-loss, and traffic light regime filter.

Reference: J.P. Morgan "RV on the EUR swap yield curve" (Apr 2021).

In [ ]:
import datetime
import itertools
import logging
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from BT.signals.regression_rv import (
    RegressionRVConfig,
    RegressionSignalTable,
    build_jpm_signal_table,
    JPMFlyUniverse,
    default_fly_universe,
)
from BT.signals.regime_filter import RegimeFilterConfig, traffic_light
from BT.signals.jpm_rv_backtest import (
    JPMRVBacktestConfig,
    JPMRVBacktestResult,
    run_jpm_rv_backtest,
)

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

%matplotlib inline
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["figure.dpi"] = 100

In [ ]:
# === CONFIGURATION ===
CURVE = "USD-SOFR-1D"
SOURCE = "ERIS_EOD_LIVE-RL_BASIC"
START_DATE = datetime.date(2020, 1, 2)
END_DATE = datetime.date(2026, 3, 20)

# Regression parameters
REG_CONFIG = RegressionRVConfig(
    window_days=130,        # 6-month rolling window
    min_rsq=0.60,
    zscore_lookback_days=130,
)

# Fly universe
UNIVERSE = default_fly_universe()
print(f"Total flies in universe: {len(UNIVERSE.all_fly_ids())}")
for cat, count in pd.Series(UNIVERSE.all_fly_ids()).value_counts().items():
    print(f"  {cat}: {count}")

In [ ]:
from TB.TimeseriesBuilder import TimeseriesBuilder
from TB.IRSwapsTB import IRSwapsTB
from Query.IRSwaps.IRSwapQuery import IRSwapQuery
from Query.IRSwaps.IRSwapStructure import IRSwapStructure
from Query.IRSwaps.IRSwapValue import IRSwapValue
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP

curve_mdp = IRSwapsMDP(source=SOURCE)
ts_builder = TimeseriesBuilder()
router = {"IRS": IRSwapsTB(curve_mdp, show_tqdm=True)}

# Build queries for all tenors needed
all_tenors = set()
for _, _, fwd, left, belly, right in UNIVERSE.iter_flies():
    for t in [left, belly, right]:
        key = t if "x" in t else (f"{fwd}x{t}" if fwd else t)
        all_tenors.add(key)

print(f"Fetching {len(all_tenors)} tenor time series...")

queries = []
for tenor in sorted(all_tenors):
    q = IRSwapQuery(
        tenor=tenor,
        curve=CURVE,
        value=IRSwapValue.RATE,
    )
    queries.append(q)

rate_df = ts_builder.get_timeseries(
    start=START_DATE, end=END_DATE, queries=queries, n_jobs=12, routers=router,
)

# Reshape into dict of tenor -> Series
rate_panels = {}
for q in queries:
    col = q.col_name()
    if col in rate_df.columns:
        rate_panels[q.tenor] = rate_df[col].dropna()
    else:
        logger.warning(f"Missing column for {q.tenor}: {col}")

print(f"Loaded {len(rate_panels)} tenor series, date range: {rate_df.index[0]} to {rate_df.index[-1]}")

In [ ]:
print("Building regression signal table...")
signal_table = build_jpm_signal_table(rate_panels, UNIVERSE, REG_CONFIG)

print(f"Successfully built signals for {len(signal_table.residuals)} flies:")
for cat in sorted(set(signal_table.fly_categories.values())):
    n = sum(1 for c in signal_table.fly_categories.values() if c == cat)
    print(f"  {cat}: {n}")

In [ ]:
# Traffic light indicator — use first gap_mm fly as reference
ref_fly_candidates = [fid for fid, cat in signal_table.fly_categories.items() if cat == "gap_mm"]
if not ref_fly_candidates:
    ref_fly_candidates = list(signal_table.fly_categories.keys())
REF_FLY = ref_fly_candidates[0]
print(f"Reference fly for traffic light: {REF_FLY}")

tl_config = RegimeFilterConfig(
    beta_vol_window_days=65,
    beta_vol_zscore_window_days=130,
    threshold=3.0,
)

tl_df = traffic_light(
    signal_table.betas_body[REF_FLY],
    signal_table.betas_curve[REF_FLY],
    tl_config,
)
regime = tl_df["regime"]

n_red = (regime == "red").sum()
n_green = (regime == "green").sum()
print(f"Traffic light: {n_green} green days, {n_red} red days ({n_red/(n_red+n_green)*100:.1f}% red)")

# Plot
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
tl_df["indicator"].plot(ax=ax1, color="black", linewidth=0.8)
ax1.axhline(tl_config.threshold, color="red", linestyle="--", alpha=0.7, label=f"Threshold={tl_config.threshold}")
ax1.fill_between(tl_df.index, 0, tl_df["indicator"],
                  where=tl_df["indicator"] > tl_config.threshold, color="red", alpha=0.3)
ax1.set_title("Traffic Light Indicator (Beta Stability)")
ax1.legend()
ax1.set_ylabel("Indicator")

signal_table.betas_body[REF_FLY].plot(ax=ax2, label="beta_body", linewidth=0.8)
signal_table.betas_curve[REF_FLY].plot(ax=ax2, label="beta_curve", linewidth=0.8)
ax2.set_title(f"Rolling Regression Betas - {REF_FLY}")
ax2.legend()
ax2.set_ylabel("Beta")
plt.tight_layout()
plt.show()

## Grid Search (JPM Exhibit 7)

In [ ]:
# === GRID SEARCH ===
RSQ_GRID = [0.60, 0.80]
RESIDUAL_BP_GRID = [2.0, 3.0, 4.0]
ZSCORE_GRID = [1.5, 2.0]

grid = list(itertools.product(RSQ_GRID, RESIDUAL_BP_GRID, ZSCORE_GRID))
print(f"Running {len(grid)} parameter combinations x 2 (with/without traffic light) = {len(grid)*2} backtests")

results_no_tl = {}
results_with_tl = {}

regime_all_green = pd.Series("green", index=regime.index)

for i, (rsq_min, res_bp, zs_min) in enumerate(grid):
    label = f"R2>={rsq_min:.0%} |res|>={res_bp}bp |z|>={zs_min}"
    config = JPMRVBacktestConfig(
        entry_min_rsq=rsq_min,
        entry_min_residual_bp=res_bp,
        entry_min_zscore=zs_min,
        exit_max_holding_days=22,
        exit_stop_loss_sd=2.0,
        trade_belly_bpv=100_000.0,
    )

    # Without traffic light
    r_no_tl = run_jpm_rv_backtest(
        signal_table=signal_table, regime=regime_all_green, mdp=None, config=config
    )
    results_no_tl[label] = r_no_tl

    # With traffic light
    r_tl = run_jpm_rv_backtest(
        signal_table=signal_table, regime=regime, mdp=None, config=config
    )
    results_with_tl[label] = r_tl

    print(f"  [{i+1}/{len(grid)}] {label}: "
          f"no_TL={r_no_tl.metrics.get('n_trades',0)} trades, Sharpe={r_no_tl.metrics.get('sharpe',0):.2f} | "
          f"TL={r_tl.metrics.get('n_trades',0)} trades, Sharpe={r_tl.metrics.get('sharpe',0):.2f}")

In [ ]:
def build_comparison_table(results_dict):
    rows = []
    for label, result in results_dict.items():
        m = result.metrics
        trades = result.trades
        rows.append({
            "Trigger": label,
            "N Trades": m.get("n_trades", 0),
            "Avg P&L (bp)": f"{m.get('avg_pnl', 0):.1f}",
            "Hit Rate": f"{m.get('hit_rate', 0):.1%}",
            "Sharpe": f"{m.get('sharpe', 0):.2f}",
            "Max DD (bp)": f"{m.get('max_drawdown', 0):.0f}",
            "Total P&L (bp)": f"{m.get('total_pnl', 0):.0f}",
            "Max Win": f"{trades['realized_pnl'].max():.1f}" if len(trades) else "N/A",
            "Max Loss": f"{trades['realized_pnl'].min():.1f}" if len(trades) else "N/A",
        })
    return pd.DataFrame(rows).set_index("Trigger")

print("=== WITHOUT TRAFFIC LIGHT ===")
display(build_comparison_table(results_no_tl))

print("\n=== WITH TRAFFIC LIGHT ===")
display(build_comparison_table(results_with_tl))

In [ ]:
# Traffic light impact table (JPM Exhibits 16-17)
impact_rows = []
for label in results_no_tl:
    m_no = results_no_tl[label].metrics
    m_tl = results_with_tl[label].metrics
    avg_no = m_no.get("avg_pnl", 0)
    avg_tl = m_tl.get("avg_pnl", 0)
    hr_no = m_no.get("hit_rate", 0)
    hr_tl = m_tl.get("hit_rate", 0)
    impact_rows.append({
        "Trigger": label,
        "Trades (no TL)": m_no.get("n_trades", 0),
        "Trades (TL)": m_tl.get("n_trades", 0),
        "Avg P&L no TL": f"{avg_no:.1f}",
        "Avg P&L with TL": f"{avg_tl:.1f}",
        "D Avg P&L": f"{avg_tl - avg_no:+.1f}",
        "Hit Rate no TL": f"{hr_no:.1%}",
        "Hit Rate with TL": f"{hr_tl:.1%}",
        "D Hit Rate": f"{hr_tl - hr_no:+.1%}",
        "Sharpe no TL": f"{m_no.get('sharpe',0):.2f}",
        "Sharpe with TL": f"{m_tl.get('sharpe',0):.2f}",
    })
display(pd.DataFrame(impact_rows).set_index("Trigger"))

## Performance Visualization

In [ ]:
# Pick the R2>=80%, res>=2bp, Z>=1.5 config as the primary analysis case
PRIMARY_LABEL = "R2>=80% |res|>=2.0bp |z|>=1.5"
primary_result = results_with_tl.get(PRIMARY_LABEL, list(results_with_tl.values())[0])
primary_no_tl = results_no_tl.get(PRIMARY_LABEL, list(results_no_tl.values())[0])

print(f"Primary config: {PRIMARY_LABEL}")
print(f"  With TL: {primary_result.metrics}")
print(f"  Without TL: {primary_no_tl.metrics}")

In [ ]:
def _contiguous_ranges(index, mask):
    """Yield (start, end) pairs for contiguous True spans in mask."""
    in_span = False
    start = None
    for i, val in enumerate(mask):
        if val and not in_span:
            in_span = True
            start = index[i]
        elif not val and in_span:
            in_span = False
            yield start, index[i - 1]
    if in_span:
        yield start, index[-1]

# Cumulative P&L with traffic light overlay (JPM Exhibit 14)
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

ax = axes[0]
primary_no_tl.cumulative_pnl.plot(ax=ax, label="Without Traffic Light", color="gray", alpha=0.7)
primary_result.cumulative_pnl.plot(ax=ax, label="With Traffic Light", color="black", linewidth=1.5)

# Shade red regime periods
red_mask = regime == "red"
if red_mask.any():
    for start, end in _contiguous_ranges(regime.index, red_mask):
        ax.axvspan(start, end, color="red", alpha=0.1)
ax.set_title(f"Cumulative P&L - {PRIMARY_LABEL}")
ax.set_ylabel("Cumulative P&L (bp equiv)")
ax.legend()
ax.grid(True, alpha=0.3)

# Bottom: Drawdown
ax2 = axes[1]
primary_result.drawdown.plot(ax=ax2, color="red", alpha=0.7)
ax2.fill_between(primary_result.drawdown.index, primary_result.drawdown, 0, color="red", alpha=0.2)
ax2.set_title("Drawdown")
ax2.set_ylabel("Drawdown (bp equiv)")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Quarterly P&L bar chart (JPM Exhibit 5)
quarterly_pnl = primary_result.daily_pnl.resample("QE").sum()

fig, ax = plt.subplots(figsize=(14, 5))
colors = ["green" if v >= 0 else "red" for v in quarterly_pnl.values]
quarterly_pnl.plot(kind="bar", color=colors, ax=ax, width=0.8)
ax.set_title(f"Quarterly P&L - {PRIMARY_LABEL} (with Traffic Light)")
ax.set_ylabel("P&L (bp equiv)")
ax.set_xticklabels([str(d)[:7] for d in quarterly_pnl.index], rotation=45, ha="right")
ax.axhline(0, color="black", linewidth=0.5)
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

# Identify critical periods (negative quarters)
critical_quarters = quarterly_pnl[quarterly_pnl < 0]
print(f"\nCritical periods ({len(critical_quarters)} negative quarters):")
for dt, pnl in critical_quarters.items():
    print(f"  {str(dt)[:7]}: {pnl:.0f} bp")

In [ ]:
# Category decomposition (JPM Exhibit 8)
if len(primary_result.trades) > 0:
    cat_stats = primary_result.trades.groupby("category").agg(
        n_trades=("realized_pnl", "count"),
        avg_pnl=("realized_pnl", "mean"),
        total_pnl=("realized_pnl", "sum"),
        hit_rate=("realized_pnl", lambda x: (x > 0).mean()),
        max_win=("realized_pnl", "max"),
        max_loss=("realized_pnl", "min"),
    ).round(2)
    print("=== Performance by Category ===")
    display(cat_stats)

    # Category cumulative P&L
    if not primary_result.daily_pnl_by_category.empty:
        fig, ax = plt.subplots(figsize=(14, 6))
        primary_result.daily_pnl_by_category.cumsum().plot(ax=ax, linewidth=1.2)
        ax.set_title("Cumulative P&L by Category")
        ax.set_ylabel("Cumulative P&L (bp equiv)")
        ax.legend(title="Category")
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

In [ ]:
# Exit reason analysis
if len(primary_result.trades) > 0:
    exit_stats = primary_result.trades.groupby("exit_reason").agg(
        n_trades=("realized_pnl", "count"),
        avg_pnl=("realized_pnl", "mean"),
        total_pnl=("realized_pnl", "sum"),
        hit_rate=("realized_pnl", lambda x: (x > 0).mean()),
        avg_holding=("holding_days", "mean"),
    ).round(2)
    print("=== Performance by Exit Reason ===")
    display(exit_stats)

    # Validation: stop-loss trades should have NEGATIVE avg P&L
    if "stop_loss" in exit_stats.index:
        sl_avg = exit_stats.loc["stop_loss", "avg_pnl"]
        print(f"\nStop-loss avg P&L: {sl_avg:.1f} (should be negative)")
        if sl_avg > 0:
            print("WARNING: Stop-loss trades showing positive P&L - investigate!")

In [ ]:
print("=== VALIDATION CHECKS ===")

# 1. No trade exceeds max holding
if len(primary_result.trades) > 0:
    max_hold = primary_result.trades["holding_days"].max()
    print(f"1. Max holding days: {max_hold} (limit: 22) - {'PASS' if max_hold <= 23 else 'FAIL'}")

# 2. Stricter criteria -> fewer trades
labels_sorted = sorted(results_with_tl.keys())
trade_counts = {l: results_with_tl[l].metrics.get("n_trades", 0) for l in labels_sorted}
print(f"2. Trade counts by trigger (stricter should be fewer):")
for l, n in trade_counts.items():
    print(f"   {l}: {n}")

# 3. Traffic light fires during stress
if red_mask.any():
    red_periods = list(_contiguous_ranges(regime.index, red_mask))
    print(f"3. Traffic light red periods ({len(red_periods)}):")
    for s, e in red_periods[:10]:
        print(f"   {s.date()} to {e.date()}")

## Full Curve-Based MTM (Optional)

Uncomment and run for accurate P&L using QueryDrivenBacktest with IRSwapsMDP.

In [ ]:
# === FULL CURVE-BASED MTM (Optional - slow but accurate) ===
# This section uses QueryDrivenBacktest with IRSwapsMDP for proper mark-to-market
# Uncomment and run when you want accurate P&L rather than approximate

# from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
# mdp_full = IRSwapsMDP(source=SOURCE)
#
# config_full = JPMRVBacktestConfig(
#     entry_min_rsq=0.80,
#     entry_min_residual_bp=2.0,
#     entry_min_zscore=1.5,
#     exit_max_holding_days=22,
#     exit_stop_loss_sd=2.0,
#     trade_belly_bpv=100_000.0,
# )
#
# result_full = run_jpm_rv_backtest(
#     signal_table=signal_table,
#     regime=regime,
#     mdp=mdp_full,
#     config=config_full,
# )
#
# print(f"Full curve MTM metrics: {result_full.metrics}")
# result_full.cumulative_pnl_ccy.plot(title="Cumulative P&L (Currency)", figsize=(14, 6))
# plt.show()